# Install all the modules

In [ ]:
%%writefile requirements.txt
transformers
torch
datasets
accelerate
peft
scikit-learn
pandas
numpy

Overwriting requirements.txt


In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -U trl

In [ ]:
!pip install PyMuPDF

In [ ]:
!pip install -q bitsandbytes

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

# Check Hugging Face Dataset

In [ ]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

# Prepare the data

In [ ]:
import pymupdf, re

In [ ]:
def split_paragraph(pages):
  para=[]
  for p in pages:
    chunks = re.split(r'\n\s*\n', p)

    para.append([i.strip() for i in chunks if len(i.strip())>=30 ])
  return para[0]





def split_by_qa_pairs(text):
    # Pattern: numbered questions jaise "1. A nurse should..."
      pattern = r'(\d+)\.\s+(.*?)(?=\n\d+\.|\Z)'
      matches = re.findall(pattern, text, re.DOTALL)

      chunks = []
      for num, content in matches:
          chunks.append({
              "text": content.strip()
          })
      return chunks


In [ ]:

pdf_path = "/content/Full.pdf"


def extract_text_from_pdf(pdf_path):
  corpus=[]
  with pymupdf.open(pdf_path) as pdf:
    for page in pdf:
      text = page.get_text('text').lower()
      text=re.sub(r"<.*?>", "", text)                      # clean the html
      text = re.sub(r"&nbsp;|&amp;|&lt;|&gt;", " ", text)  # common entities
      # text = re.sub(r"\s+", " ", text).strip()             # remove extra spaces
      if text:
        corpus.append(text.strip())
        full_text = "\n".join(corpus)
  return split_by_qa_pairs(full_text)





text_dict=extract_text_from_pdf(pdf_path)
text_dict


[{'text': 'any opinions, findings, and conclusions or recommendations expressed in this material are those of the\nauthor(s) and do not necessarily reflect the views of the national science foundation nor the us\ndepartment of education.\nhave \nquestions \nor \ncomments? \nfor \ninformation \nabout \nadoptions \nor \nadaptions \ncontact\ninfo@libretexts.org or visit our main website at https://libretexts.org.\nthis text was compiled on 08/12/2025\n1\nhttps://med.libretexts.org/@go/page/24347\nintroduction\nthis open access nursing pharmacology textbook is designed for entry-level undergraduate nursing students. it explains basic\nconcepts of pharmacology and describes common medication classes. this book is not intended to be used as a drug reference\nbook, but direct links are provided to dailymed, which provides trustworthy information about marketed drugs in the united\nstates.\nthis textbook is aligned with the wisconsin technical college system (wtcs) statewide nursing curriculum

In [ ]:
# text_dict=[]
# for text in doc_list:
#   text_dict.append({'text': text})
# text_dict[0]



In [ ]:
dataset_met=Dataset.from_list(text_dict)
dataset_met[0]

{'text': 'any opinions, findings, and conclusions or recommendations expressed in this material are those of the\nauthor(s) and do not necessarily reflect the views of the national science foundation nor the us\ndepartment of education.\nhave \nquestions \nor \ncomments? \nfor \ninformation \nabout \nadoptions \nor \nadaptions \ncontact\ninfo@libretexts.org or visit our main website at https://libretexts.org.\nthis text was compiled on 08/12/2025\n1\nhttps://med.libretexts.org/@go/page/24347\nintroduction\nthis open access nursing pharmacology textbook is designed for entry-level undergraduate nursing students. it explains basic\nconcepts of pharmacology and describes common medication classes. this book is not intended to be used as a drug reference\nbook, but direct links are provided to dailymed, which provides trustworthy information about marketed drugs in the united\nstates.\nthis textbook is aligned with the wisconsin technical college system (wtcs) statewide nursing curriculum 

# Define the Model we are going to Use

In [16]:
# model_name = "meta-llama/Llama-2-7b-hf"
# model_name = "meta-llama/Llama-3.1-8B"
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [20]:
tokenizer= AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [21]:
tokenizer

LlamaTokenizer(name_or_path='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', vocab_size=32000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

# If padding is not given we need to use end of token as padding


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize the entire data set

In [ ]:
def mass_tokenizer(example):
  tokens=tokenizer(example['text'],  max_length = 512, padding = 'max_length')
  tokens['labels']=tokens['input_ids'].copy()

  return tokens


encoded_dataset=dataset_met.map(mass_tokenizer, batched=True, remove_columns=['text'])
encoded_dataset

Map:   0%|          | 0/1051 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1051
})

# Define the training argumnets

In [ ]:
from transformers import TrainingArguments, Trainer
import torch
device= ('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

args= TrainingArguments(
    output_dir ='./llma_pharma_domain',
    per_device_train_batch_size = 8,
    num_train_epochs = 2,
    learning_rate = 2e-5 ,
    weight_decay = 0.01,
    save_strategy = 'epoch',
    logging_strategy = 'epoch',
    save_steps=500,
    save_total_limit=2

)

cuda


# Try to do the full fine tuning 🤣🤣🤣🤣

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset,

)

OutOfMemoryError: CUDA out of memory. Tried to allocate 250.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 27.81 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 14.02 GiB is allocated by PyTorch, and 393.10 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

# Out of Memory

In [ ]:
trainer.train().to(device)

# LoRA

In [ ]:

import torch, gc

# gc ==> Garbedge collector : this will remove the casche memory
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType



In [ ]:

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
!pip install --upgrade torchao
non_inst_model_lora = get_peft_model(model, lora_config)

In [ ]:

from transformers import TrainingArguments, Trainer
import torch
device= ('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
args = TrainingArguments(
    output_dir="./tinyllama-lora_nursing",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

cuda


In [ ]:
encoded_dataset.select(range(100))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

In [ ]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=encoded_dataset.select(range(100))
)

In [ ]:
trainer.train()

Step,Training Loss
20,0.344046
40,0.304391
60,0.310794


Streaming output truncated to the last 5000 lines.


TrainOutput(global_step=65, training_loss=0.3126507355616643, metrics={'train_runtime': 409.2704, 'train_samples_per_second': 1.222, 'train_steps_per_second': 0.159, 'total_flos': 1599658022154240.0, 'train_loss': 0.3126507355616643, 'epoch': 5.0})

In [8]:
model_path = "/content/tinyllama-lora_nursing/checkpoint-65"

In [9]:

from transformers import AutoTokenizer, AutoModelForCausalLM


model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [25]:
prompt = "Pharmacokinetics is the term that describes the four stages of absorption, distribution, metabolism"

In [26]:
# !pip install sentencepiece
tokenizer = AutoTokenizer.from_pretrained(model_name)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")


In [27]:

outputs = model.generate(
    **inputs,
    max_new_tokens=1000,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [28]:

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Pharmacokinetics is the term that describes the four stages of absorption, distribution, metabolism and excretion of a drug

### Exercise 3.15: Focusing on Pharmacokinetics in Clinical Practice

**Instructor:** Welcome to this week's exercise. Let’s begin by reviewing the process of pharmacokinetics.

**Participants:** Students and instructor.

**Instructor:** Okay, let’s start with the first point that we have discussed in class. Pharmacokinetic parameters are based on the concept of “first-pass”

this means that drugs can be metabolized or broken down inside the body before reaching the bloodstream.

**Students:** Okay, this sounds like a good idea. Why don’t you stop me when I’m ready?

**Instructor:** Okay, let’s do it! When a drug reaches the bloodstream, it will undergo extensive interactions

with the liver and other organs within the human body. In fact, most drugs reach the liver and go through this stage before they reach

the bloodstream. This is why it is im